# Construção dos Indicadores

Nesta etapa são construídos os indicadores consolidados a partir do dataset integrado, com o objetivo de disponibilizar tabelas analíticas para validação dos resultados e utilização posterior no Power BI.

Os indicadores contemplam:

* visão geral da base;
* produtividade por cultura;
* evolução temporal;
* produtividade por país;
* rankings de países por cultura;
* associações entre produtividade e variáveis ambientais e de produção;
* indicadores de qualidade e cobertura.

As tabelas geradas nesta etapa serão exportadas em formato CSV para a pasta `dados/indicadores/`.


In [ ]:
import pandas as pd

yield_integrado = pd.read_csv('../dados/yield_integrado.csv')
yield_integrado.info()

## 1. Indicadores gerais

In [ ]:
indicadores_gerais = {
    'registros': len(yield_integrado),
    'paises': yield_integrado['Area'].nunique(),
    'culturas': yield_integrado['Item'].nunique(),
    'ano_inicial': yield_integrado['Year'].min(),
    'ano_final': yield_integrado['Year'].max(),
    'produtividade_mediana': yield_integrado['Value'].median(),
    'produtividade_media': yield_integrado['Value'].mean(),
    'produtividade_minima': yield_integrado['Value'].min(),
    'produtividade_maxima': yield_integrado['Value'].max()
}

pd.Series(indicadores_gerais)

## 2. Indicadores por cultura

In [ ]:
indicadores_cultura = (
    yield_integrado
    .groupby('Item')
    .agg(
        registros=('Value', 'count'),
        paises=('Area', 'nunique'),
        produtividade_mediana=('Value', 'median'),
        produtividade_media=('Value', 'mean'),
        produtividade_minima=('Value', 'min'),
        produtividade_maxima=('Value', 'max')
    )
    .sort_values('produtividade_mediana', ascending=False)
)

indicadores_cultura

## 3. Indicadores temporais

In [ ]:
produtividade_inicio = (
    yield_integrado[yield_integrado['Year'] == 1961]
    .groupby('Item')['Value']
    .median()
    .rename('produtividade_1961')
)

produtividade_fim = (
    yield_integrado[yield_integrado['Year'] == 2016]
    .groupby('Item')['Value']
    .median()
    .rename('produtividade_2016')
)

indicadores_temporais = pd.concat(
    [produtividade_inicio, produtividade_fim],
    axis=1
)

indicadores_temporais['variacao_absoluta'] = (
    indicadores_temporais['produtividade_2016']
    - indicadores_temporais['produtividade_1961']
)

indicadores_temporais['crescimento_percentual'] = (
    (
        indicadores_temporais['produtividade_2016']
        / indicadores_temporais['produtividade_1961']
        - 1
    ) * 100
)

indicadores_temporais.sort_values(
    'crescimento_percentual',
    ascending=False
)

### 3.1 Indicadores temporais anuais

In [ ]:
indicadores_temporais_anuais = (
    yield_integrado
    .groupby(['Year', 'Item'])
    .agg(
        registros=('Value', 'count'),
        paises=('Area', 'nunique'),
        produtividade_mediana=('Value', 'median'),
        produtividade_media=('Value', 'mean'),
        produtividade_minima=('Value', 'min'),
        produtividade_maxima=('Value', 'max')
    )
    .reset_index()
    .sort_values(['Item', 'Year'])
)

indicadores_temporais_anuais.head(20)

## 4. Indicadores por país

In [ ]:
indicadores_pais = (
    yield_integrado
    .groupby(['Item', 'Area'])
    .agg(
        registros=('Value', 'count'),
        produtividade_mediana=('Value', 'median'),
        produtividade_media=('Value', 'mean'),
        produtividade_minima=('Value', 'min'),
        produtividade_maxima=('Value', 'max')
    )
    .reset_index()
)

indicadores_pais = (
    indicadores_pais[indicadores_pais['registros'] >= 20]
    .sort_values(
        ['Item', 'produtividade_mediana'],
        ascending=[True, False]
    )
)

indicadores_pais.head(20)

## 5. Ranking de países por cultura

In [ ]:
top3_paises = (
    indicadores_pais
    .groupby('Item', group_keys=False)
    .head(3)
    .assign(posicao='Top 3')
)

bottom3_paises = (
    indicadores_pais
    .groupby('Item', group_keys=False)
    .tail(3)
    .sort_values(['Item', 'produtividade_mediana'], ascending=[True, True])
    .assign(posicao='Bottom 3')
)

ranking_paises = pd.concat(
    [top3_paises, bottom3_paises]
)

ranking_paises

## 6. Indicadores de associação

In [ ]:
def calcular_correlacao(coluna):
    return (
        yield_integrado[
            ['Item', 'Value', coluna]
        ]
        .dropna(subset=[coluna])
        .groupby('Item')
        .apply(
            lambda grupo: grupo['Value'].corr(grupo[coluna]),
            include_groups=False
        )
        .rename('correlacao')
        .reset_index()
    )

correlacao_temp = calcular_correlacao('avg_temp')
correlacao_chuva = calcular_correlacao('rainfall_mm')
correlacao_pesticidas = calcular_correlacao('pesticides_tonnes')

indicadores_associacao = (
    correlacao_temp
    .rename(columns={'correlacao': 'correlacao_temperatura'})
    .merge(
        correlacao_chuva.rename(
            columns={'correlacao': 'correlacao_chuva'}
        ),
        on='Item',
        how='outer'
    )
    .merge(
        correlacao_pesticidas.rename(
            columns={'correlacao': 'correlacao_pesticidas'}
        ),
        on='Item',
        how='outer'
    )
)

indicadores_associacao

## 7. Indicadores de qualidade

In [ ]:
indicadores_qualidade = {
    'valores_zero': (yield_integrado['Value'] == 0).sum(),
    'valores_abaixo_1000': (yield_integrado['Value'] < 1000).sum(),
    'valores_acima_500000': (yield_integrado['Value'] > 500000).sum(),
    'valores_nulos_produtividade': yield_integrado['Value'].isna().sum(),
    'paises': yield_integrado['Area'].nunique(),
    'culturas': yield_integrado['Item'].nunique(),
    'registros': len(yield_integrado)
}

pd.Series(indicadores_qualidade)

## 8. Validações para o Power BI

In [ ]:
validacoes_power_bi = {
    'indicadores_temporais_duplicados': indicadores_temporais.reset_index()['Item'].duplicated().sum(),
    'indicadores_temporais_anuais_duplicados': indicadores_temporais_anuais.duplicated(['Year', 'Item']).sum(),
    'indicadores_pais_duplicados': indicadores_pais.duplicated(['Item', 'Area']).sum(),
    'ranking_paises_duplicados': ranking_paises.duplicated(['Item', 'Area']).sum(),
    'indicadores_associacao_duplicados': indicadores_associacao['Item'].duplicated().sum(),
    'itens_base': yield_integrado['Item'].nunique(),
    'itens_temporais': indicadores_temporais.index.nunique(),
    'itens_temporais_anuais': indicadores_temporais_anuais['Item'].nunique(),
    'itens_associacao': indicadores_associacao['Item'].nunique(),
    'paises_base': yield_integrado['Area'].nunique(),
    'paises_indicadores': indicadores_pais['Area'].nunique()
}

pd.Series(validacoes_power_bi)

## 9. Exportação dos indicadores

In [ ]:
import os

os.makedirs('../dados/indicadores', exist_ok=True)

tabelas_exportacao = {
    'indicadores_temporais': indicadores_temporais.reset_index(),
    'indicadores_temporais_anuais': indicadores_temporais_anuais,
    'indicadores_pais': indicadores_pais,
    'indicadores_associacao': indicadores_associacao,
    'ranking_paises': ranking_paises
}

for nome, tabela in tabelas_exportacao.items():
    caminho = f'../dados/indicadores/{nome}.csv'
    tabela.to_csv(caminho, index=False)
    print(f'{nome}.csv exportado: {tabela.shape}')